In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.impute import SimpleImputer

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# --------------------------------------------------
# 1. Load Dataset
# --------------------------------------------------

df = pd.read_csv("/content/placement_predict_50k Dataset (1).csv")

print("FIRST 5 ROWS")
print(df.head())


# --------------------------------------------------
# 2. Remove Duplicates
# --------------------------------------------------

df = df.drop_duplicates()

print("\nDATASET SHAPE")
print(df.shape)


# --------------------------------------------------
# 3. Select Target
# --------------------------------------------------

target_column = "PlacementStatus"

if target_column not in df.columns:

    print("\nPlacement column not found.")

    print("Available columns:")
    print(df.columns.tolist())

else:

    X = df.drop(columns=[target_column])
    y = df[target_column]


    # --------------------------------------------------
    # 4. Convert Target if Necessary
    # --------------------------------------------------

    if y.dtype == "object":

        y = y.astype(str).str.lower()

        mapping = {
            "yes": 1,
            "no": 0,
            "placed": 1,
            "not placed": 0,
            "true": 1,
            "false": 0
        }

        y = y.map(mapping)

    y = pd.to_numeric(
        y,
        errors="coerce"
    )

    valid_rows = y.notna()

    X = X.loc[valid_rows]
    y = y.loc[valid_rows]


    # --------------------------------------------------
    # 5. Identify Columns
    # --------------------------------------------------

    numerical_columns = X.select_dtypes(
        include=["int64", "float64"]
    ).columns.tolist()

    categorical_columns = X.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()


    print("\nNUMERICAL COLUMNS")
    print(numerical_columns)

    print("\nCATEGORICAL COLUMNS")
    print(categorical_columns)


    # --------------------------------------------------
    # 6. Numerical Pipeline
    # --------------------------------------------------

    numerical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "scaler",
                StandardScaler()
            )
        ]
    )


    # --------------------------------------------------
    # 7. Categorical Pipeline
    # --------------------------------------------------

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent")
            ),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore"
                )
            )
        ]
    )


    # --------------------------------------------------
    # 8. Column Transformer
    # --------------------------------------------------

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                numerical_pipeline,
                numerical_columns
            ),

            (
                "cat",
                categorical_pipeline,
                categorical_columns
            )
        ]
    )


    # --------------------------------------------------
    # 9. Create Logistic Regression Model
    # --------------------------------------------------

    model = LogisticRegression(
        max_iter=1000
    )


    # --------------------------------------------------
    # 10. Complete Pipeline
    # --------------------------------------------------

    pipeline = Pipeline(
        steps=[
            (
                "preprocessing",
                preprocessor
            ),

            (
                "classification",
                model
            )
        ]
    )


    # --------------------------------------------------
    # 11. Train-Test Split
    # --------------------------------------------------

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )


    # --------------------------------------------------
    # 12. Train Model
    # --------------------------------------------------

    pipeline.fit(
        X_train,
        y_train
    )


    # --------------------------------------------------
    # 13. Predictions
    # --------------------------------------------------

    y_pred = pipeline.predict(X_test)

    y_probability = pipeline.predict_proba(
        X_test
    )[:, 1]


    # --------------------------------------------------
    # 14. Evaluation
    # --------------------------------------------------

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )


    print("\nMODEL PERFORMANCE")

    print("Accuracy :", accuracy)
    print("Precision:", precision)
    print("Recall   :", recall)
    print("F1 Score :", f1)


    # --------------------------------------------------
    # 15. Confusion Matrix
    # --------------------------------------------------

    print("\nCONFUSION MATRIX")

    print(
        confusion_matrix(
            y_test,
            y_pred
        )
    )


    # --------------------------------------------------
    # 16. Classification Report
    # --------------------------------------------------

    print("\nCLASSIFICATION REPORT")

    print(
        classification_report(
            y_test,
            y_pred,
            zero_division=0
        )
    )


    # --------------------------------------------------
    # 17. Probability Output
    # --------------------------------------------------

    print("\nFIRST 10 PREDICTION PROBABILITIES")

    print(y_probability[:10])


    print("\nEXPERIMENT 5 COMPLETED")

FIRST 5 ROWS
   StudentID  Gender       City CollegeTier Stream Specialisation Hostel  \
0          1    Male  Ahmedabad       Tier2    ECE     Networking     No   
1          2  Female     Mumbai       Tier2    ECE    DataScience    Yes   
2          3    Male    Kolkata       Tier2     IT    DataScience    Yes   
3          4    Male     Jaipur       Tier1     CS             AI     No   
4          5    Male       Pune       Tier2     IT    DataScience    Yes   

  HistoryOfBacklogs  SGPA_Sem1  SGPA_Sem2  ...  Publications  \
0                No       6.02       6.54  ...             0   
1               Yes       5.84       5.12  ...             0   
2                No       4.91       5.29  ...             0   
3                No       7.67       8.03  ...             0   
4                No       8.14       8.97  ...             1   

   AptitudeTestScore  SoftSkillsRating  CodingTestScore  MockInterviewScore  \
0               66.7               2.2             49.4           